# Упражнение: YOLO: декодирование сетки и якорей, порог уверенности и NMS

## Задание

YOLO-подобный детектор (в стиле YOLOv3) для картинки 128 × 128 выдал «сырой» тензор на сетке 4 × 4
(шаг stride = 32 пикселя) с 2 якорями на клетку. Файл `input.csv` — 32 строки
`gy, gx, anchor, tx, ty, tw, th, obj, cls_person, cls_car` (все значения — логиты/сырые выходы).
Параметры — в `config.json`: размеры якорей `anchors` (ширина, высота в пикселях), классы, пороги.

Декодирование (σ — сигмоида):

- центр: bx = (σ(tx) + gx) · stride, by = (σ(ty) + gy) · stride;
- размер: bw = p_w · exp(tw), bh = p_h · exp(th), где (p_w, p_h) — якорь `anchor`;
- класс — с максимальным логитом; уверенность score = σ(obj) · σ(логит этого класса);
- рамка в углах: x1 = bx − bw/2, y1 = by − bh/2, x2 = bx + bw/2, y2 = by + bh/2, затем **обрезать** по
  картинке [0, 128].

Постобработка:

1. Оставить рамки со score **> 0.25** (`score_threshold`).
2. NMS **по каждому классу отдельно**: по убыванию score берём лучшую рамку и удаляем рамки того же класса
   с IoU **> 0.5** с ней (IoU — по обрезанным рамкам, площадь (x2 − x1)·(y2 − y1)).

## Подсказки

- Сигмоида держит центр внутри «своей» клетки, exp — делает размер положительным и масштабирует якорь.
- Почти все 32 предсказания — фон с очень маленьким σ(obj). После порога остаётся горстка рамок, и NMS
  схлопывает дубли одного объекта из соседних клеток и якорей.
- `cx = (sig(tx) + gx) * stride`, `w = anchors[anchor][0] * np.exp(tw)`; NMS — как в упражнении
  «IoU и NMS»: `order.pop(0)` и фильтр `iou <= 0.5`.

## Ожидаемый формат выходного файла

CSV с колонками `gy, gx, anchor, cls, score, x1, y1, x2, y2` — оставленные рамки по убыванию `score`;
`score` округлить до 4 знаков, координаты — до 2.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/yolo
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.